In [1]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [2]:
import numpy as np
import pandas as pd
import requests
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [7]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 47.5788221,
	"longitude": -122.4112033,
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation_probability", "precipitation", "rain", "is_day"],
	"timezone": "America/Los_Angeles",
	"past_days": 2,
	"wind_speed_unit": "mph",
	"temperature_unit": "fahrenheit",
	"precipitation_unit": "inch",
	"forecast_hours": 12,
	"past_hours": 6,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(4).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(5).ValuesAsNumpy()
hourly_rain = hourly.Variables(6).ValuesAsNumpy()
hourly_is_day = hourly.Variables(7).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["is_day"] = hourly_is_day

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 47.555702209472656°N -122.38996887207031°E
Elevation: 0.0 m asl
Timezone: b'America/Los_Angeles'b'GMT-7'
Timezone difference to GMT+0: -25200s

Hourly data
                         date  temperature_2m  relative_humidity_2m  \
0  2026-03-23 10:00:00+00:00       42.460701                  89.0   
1  2026-03-23 11:00:00+00:00       44.530701                  86.0   
2  2026-03-23 12:00:00+00:00       45.790699                  80.0   
3  2026-03-23 13:00:00+00:00       45.970699                  69.0   
4  2026-03-23 14:00:00+00:00       46.600700                  70.0   
5  2026-03-23 15:00:00+00:00       46.870701                  70.0   
6  2026-03-23 16:00:00+00:00       47.050697                  70.0   
7  2026-03-23 17:00:00+00:00       47.320702                  75.0   
8  2026-03-23 18:00:00+00:00       46.780701                  78.0   
9  2026-03-23 19:00:00+00:00       45.070702                  81.0   
10 2026-03-23 20:00:00+00:00       43.810699                

In [3]:
import sys
sys.path.insert(0, "..")
from src.weather_client import WeatherClient

In [4]:
client = WeatherClient()


In [5]:
client.get_forecast_df()

,time,temperature_2m,apparent_temperature,relativehumidity_2m,windspeed_10m,precipitation,cloudcover,precipitation_probability
0,2026-03-26 00:00:00,37.7,32.5,77,2.8,0.0,31,4
1,2026-03-26 01:00:00,36.6,29.9,79,6.2,0.0,0,5
2,2026-03-26 02:00:00,36.3,28.8,83,8.5,0.0,5,3
3,2026-03-26 03:00:00,35.8,30.6,88,3.4,0.0,2,0
4,2026-03-26 04:00:00,35.0,29.1,89,4.8,0.0,4,0
...,...,...,...,...,...,...,...,...
163,2026-04-01 19:00:00,42.8,38.4,93,5.4,0.0,99,60
164,2026-04-01 20:00:00,42.2,37.9,93,4.6,0.0,99,60
165,2026-04-01 21:00:00,41.4,37.0,94,4.5,0.0,99,60
166,2026-04-01 22:00:00,40.6,36.2,95,4.4,0.0,100,60
